[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/pydantic-certified/notebooks/day-02-field-types-defaults.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Fields, Types, and Defaults — Controlling Your Schema
**certified-journeys / pydantic-certified** · Day 2 · Schema Design

> **Goal for today:** By the end of this notebook you can use `Field()` with metadata and constraints, apply annotated built-in types (`PositiveInt`, `constr`, `conlist`), add aliases, and understand strict vs lax mode.

In [ ]:
%pip install -q 'pydantic>=2.0' pydantic-settings

## Step 1 · Field() — required, defaults, and metadata

`Field()` is a factory that attaches metadata to a field annotation. The first positional
argument (or `default=`) sets the default; using `...` (Ellipsis) marks the field as
**explicitly required** (same as having no default, but makes intent visible).

| Syntax | Required? | Notes |
|--------|-----------|-------|
| `age: int` | Yes | bare annotation |
| `age: int = Field(...)` | Yes | explicit required |
| `age: int = Field(default=18)` | No | scalar default |
| `tags: list = Field(default_factory=list)` | No | factory for mutable defaults |

In [ ]:
from pydantic import BaseModel, Field

class Article(BaseModel):
    # Ellipsis ("...") means required — identical behaviour to bare annotation
    title: str = Field(..., description="Article headline, max 120 chars")

    # Scalar default — safe because int is immutable
    view_count: int = Field(default=0, ge=0, description="Never negative")

    # default_factory — ALWAYS use for mutable defaults (list, dict, set)
    # Using `default=[]` would share the same list across all instances!
    tags: list[str] = Field(default_factory=list, description="Content tags")

# Create an article — only title is required
a1 = Article(title="Pydantic v2 is fast")
print(a1)

# Confirm tags are independent lists, not shared
a2 = Article(title="Another article")
a1.tags.append("pydantic")
print("a1 tags:", a1.tags)
print("a2 tags (should be empty):", a2.tags)

**What just happened?**

- `Field(...)` and a bare annotation are **equivalent for requiredness** — use `Field(...)` when you want to add metadata alongside.
- `default_factory=list` creates a fresh `[]` for every new instance — the classic Python mutable-default trap is avoided.
- **`ge=0`** is a constraint shorthand for "greater than or equal to 0" — we'll explore more constraints in Step 3.
- `description=` shows up in the JSON schema, which FastAPI uses for OpenAPI docs.

## Step 2 · Built-in constrained types: PositiveInt, constr, conlist

Pydantic ships annotated types that encode common constraints directly in the type:

| Type | Constraint |
|------|------------|
| `PositiveInt` | `int > 0` |
| `NegativeInt` | `int < 0` |
| `NonNegativeInt` | `int >= 0` |
| `PositiveFloat` | `float > 0` |
| `constr(min_length=1, max_length=50)` | constrained string |
| `conlist(str, min_length=1)` | constrained list |

These are just `Annotated[base_type, constraints]` under the hood.

In [ ]:
from pydantic import BaseModel, Field
from pydantic import PositiveInt, constr, conlist
from pydantic import ValidationError

class Product(BaseModel):
    # PositiveInt rejects 0 and negatives
    quantity: PositiveInt

    # constr constrains string length and can enforce a regex pattern
    sku: constr(min_length=3, max_length=20, pattern=r'^[A-Z0-9-]+$')

    # conlist requires at least 1 item
    categories: conlist(str, min_length=1)

# Valid product
p = Product(quantity=5, sku="WIDGET-001", categories=["hardware", "tools"])
print("Valid product:", p)

# quantity=0 violates PositiveInt
try:
    Product(quantity=0, sku="ABC-1", categories=["x"])
except ValidationError as e:
    for err in e.errors():
        print(f"  Error on {err['loc']}: {err['msg']}")

# sku with lowercase violates pattern
try:
    Product(quantity=1, sku="lowercase", categories=["x"])
except ValidationError as e:
    for err in e.errors():
        print(f"  Error on {err['loc']}: {err['msg']}")

# empty categories list violates conlist min_length
try:
    Product(quantity=1, sku="ABC-1", categories=[])
except ValidationError as e:
    for err in e.errors():
        print(f"  Error on {err['loc']}: {err['msg']}")

**What just happened?**

- `PositiveInt`, `constr`, and `conlist` are **annotated type aliases** — they encode constraints in the type, not in the model.
- `constr(pattern=...)` uses Python `re` syntax — anchor with `^...$` to match the full string.
- **Constraints appear in the JSON schema** — external validators and OpenAPI clients benefit automatically.
- Each violation shows up as a separate error in the `ValidationError`.

## Step 3 · Field constraints inline: ge, le, gt, lt, min_length, max_length

As an alternative to `constr`/`PositiveInt`, you can pass numeric and string constraints
directly to `Field()`:

| Arg | Meaning |
|-----|---------|
| `gt=x` | greater than x |
| `ge=x` | greater than or equal to x |
| `lt=x` | less than x |
| `le=x` | less than or equal to x |
| `min_length=n` | min chars/items |
| `max_length=n` | max chars/items |
| `pattern=r'...'` | regex for strings |
| `multiple_of=n` | numeric multiple |

In [ ]:
from pydantic import BaseModel, Field, ValidationError

class Order(BaseModel):
    # Price must be > 0 and rounded to 2 decimal places is enforced by the caller;
    # here we just ensure it's positive
    price: float = Field(..., gt=0, description="Price in USD, must be positive")

    # Discount is 0–100 inclusive
    discount_pct: float = Field(default=0.0, ge=0.0, le=100.0)

    # Reference code: 6–12 alphanumeric chars
    ref_code: str = Field(..., min_length=6, max_length=12)

# Valid order
o = Order(price=99.99, discount_pct=10.0, ref_code="ORD123456")
print("Valid order:", o)

# Multiple violations at once
try:
    Order(price=-5.0, discount_pct=150.0, ref_code="X")
except ValidationError as e:
    print(f"\n{e.error_count()} errors found:")
    for err in e.errors():
        print(f"  [{err['loc'][0]}] {err['msg']}")

**What just happened?**

- Inline `Field()` constraints and annotated types (`PositiveInt`, `constr`) produce **identical JSON schema output** — choose whichever reads more clearly.
- All violated constraints are **reported together** in one `ValidationError`.
- `gt=0` (strictly greater) vs `ge=0` (greater or equal) is a common source of subtle bugs — be deliberate.
- `description=` strings flow into OpenAPI documentation when used with FastAPI.

## Step 4 · Aliases: alias and validation_alias

Aliases let your Python model use idiomatic names while accepting data from external sources
that use different conventions (e.g. camelCase JSON, legacy snake_case keys).

| Parameter | Effect |
|-----------|--------|
| `alias="camelCase"` | Input and output both use the alias |
| `validation_alias="camelCase"` | Input uses alias; Python attribute keeps original name |
| `serialization_alias="camelCase"` | Output uses alias; input keeps original name |

In [ ]:
from pydantic import BaseModel, Field

class UserProfile(BaseModel):
    # validation_alias: accept "firstName" from JSON; expose as .first_name in Python
    first_name: str = Field(..., validation_alias="firstName")
    last_name: str = Field(..., validation_alias="lastName")

    # alias: both input AND output use "userAge"
    user_age: int = Field(..., alias="userAge")

# model_validate with by_alias-style keys (matching validation_alias / alias)
raw_json = {"firstName": "Grace", "lastName": "Hopper", "userAge": 85}
profile = UserProfile.model_validate(raw_json)

# Python attributes use the Pythonic names
print("first_name:", profile.first_name)
print("last_name:",  profile.last_name)
print("user_age:",   profile.user_age)

# model_dump(by_alias=True) serialises using alias names
print("\nDump with aliases:", profile.model_dump(by_alias=True))

# model_dump() without by_alias uses Python attribute names
print("Dump without aliases:", profile.model_dump())

**What just happened?**

- `validation_alias` is the most common choice for consuming camelCase APIs — Python attributes stay Pythonic.
- `alias` affects *both* input validation and output serialisation — be careful when using with `model_dump()`.
- `model_dump(by_alias=True)` is needed to round-trip data back to camelCase JSON.
- **AliasPath and AliasChoices** (advanced — Day 4+) let you alias deeply nested keys.

## Step 5 · Strict mode vs lax mode

By default, Pydantic runs in **lax mode**: it coerces compatible types (e.g. `"30"` → `int`).
**Strict mode** disables all coercion — the input type must match the annotation exactly.

Enable strict mode at three levels:

| Level | How |
|-------|-----|
| Whole model | `model_config = ConfigDict(strict=True)` |
| Single field | `Field(..., strict=True)` |
| Single call | `model_validate(data, strict=True)` |

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from pydantic import ConfigDict

# --- Model-level strict mode ---
class StrictUser(BaseModel):
    model_config = ConfigDict(strict=True)
    name: str
    age: int

# This passes in lax mode but fails in strict mode
try:
    StrictUser(name="Hiro", age="30")  # "30" is str, not int
except ValidationError as e:
    print("Strict model error:", e.errors()[0]["msg"])

# --- Per-field strict mode (rest of model is lax) ---
class HybridModel(BaseModel):
    name: str               # lax — "42" → "42" OK
    count: int = Field(..., strict=True)  # strict — "5" → ValidationError

try:
    HybridModel(name="test", count="5")  # count must be int, not str
except ValidationError as e:
    print("Hybrid field error:", e.errors()[0]["msg"])

# lax name field still coerces
h = HybridModel(name=99, count=5)
print("name coerced to str:", type(h.name), h.name)

# --- Per-call strict mode ---
class LaxUser(BaseModel):
    name: str
    age: int

try:
    LaxUser.model_validate({"name": "Kai", "age": "20"}, strict=True)
except ValidationError as e:
    print("Per-call strict error:", e.errors()[0]["msg"])

**What just happened?**

- Strict mode is useful for **internal service-to-service data** where you control both ends and coercion would hide bugs.
- Per-field strict via `Field(strict=True)` gives surgical control — e.g. strict on IDs, lax on user-facing strings.
- `model_validate(..., strict=True)` is great for **testing** — confirm your production data is already the right type.
- Strict mode does not affect *validators* you write with `@field_validator` — those run after type checks.

## Step 6 · Annotated[] with Field() for reusable constrained types

`Annotated[T, Field(...)]` lets you define a constrained type **once** and reuse it
across multiple models — a DRY alternative to repeating `Field()` in every class.

```python
from typing import Annotated
from pydantic import Field

# Define once
PositivePrice = Annotated[float, Field(gt=0, description="Price in USD")]

# Reuse in any model
class Order(BaseModel):
    price: PositivePrice

class Quote(BaseModel):
    unit_price: PositivePrice
```

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

# ── Reusable type aliases ────────────────────────────────────────────────────

# A non-empty string with max 100 chars
ShortStr = Annotated[str, Field(min_length=1, max_length=100)]

# A price that must be positive
PositivePrice = Annotated[float, Field(gt=0.0, description="Amount in USD")]

# A percentage 0–100
Percentage = Annotated[float, Field(ge=0.0, le=100.0)]

# ── Models that reuse the aliases ───────────────────────────────────────────

class Product(BaseModel):
    name: ShortStr
    price: PositivePrice
    tax_rate: Percentage = 0.0

class LineItem(BaseModel):
    description: ShortStr
    unit_price: PositivePrice
    discount: Percentage = 0.0

# Both models share the same constraint logic
p = Product(name="Widget", price=9.99, tax_rate=8.5)
li = LineItem(description="Widget x10", unit_price=9.99, discount=5.0)
print("Product:", p)
print("LineItem:", li)

# Constraint violation uses the reusable alias
try:
    Product(name="", price=-1.0, tax_rate=110.0)
except ValidationError as e:
    print(f"\n{e.error_count()} errors:")
    for err in e.errors():
        print(f"  [{err['loc'][0]}] {err['msg']}")

**What just happened?**

- `Annotated[T, Field(...)]` separates the *type* from the *constraints* — both are reusable independently.
- **One change to `PositivePrice` updates every model that uses it** — this is the key advantage over repeating `Field(gt=0)` everywhere.
- These aliases work with any Pydantic construct: model fields, function parameters via `validate_call`, and FastAPI path/query params.
- The constraint metadata also flows into the generated JSON schema.

In [ ]:
# Challenge: Build a BlogPost model using Annotated reusable types
#
# 1. Define these reusable type aliases:
#    - Title: Annotated[str, Field(min_length=5, max_length=120)]
#    - Slug:  Annotated[str, Field(pattern=r'^[a-z0-9-]+$', max_length=80)]
#    - WordCount: Annotated[int, Field(ge=100)]
#
# 2. Create a BlogPost model with fields:
#    - title: Title
#    - slug: Slug
#    - word_count: WordCount
#    - published: bool = False
#    - author_name: str = Field(..., validation_alias="authorName")
#
# 3. Validate this dict:
#    {"authorName": "Ada Lovelace", "title": "On the Analytical Engine",
#     "slug": "on-the-analytical-engine", "word_count": 2500}
#
# 4. Print model_dump(by_alias=False) and model_json_schema()
#
# Your solution here


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| `Field(...)` | Ellipsis = required; adds metadata (description, constraints) |
| `default_factory` | Always use for mutable defaults (list, dict, set) |
| `PositiveInt`, `constr`, `conlist` | Annotated type aliases for common constraints |
| Inline constraints | `ge`, `le`, `gt`, `lt`, `min_length`, `max_length`, `pattern` |
| `validation_alias` | Accept external key names; Python attr keeps original name |
| `alias` | Both input and output use the alias |
| Lax mode (default) | Compatible types coerced automatically |
| Strict mode | No coercion — exact type match required |
| `Annotated[T, Field(...)]` | DRY reusable constrained types across models |

> **Tip:** Prefer `Annotated[T, Field(...)]` over class-level `Field()` assignments for reusable constrained types — you can share them across models with a simple type alias.

---
## What's next
**Day 3** → Field Validators — use `@field_validator` with `mode='before'`/`'after'` to add custom transformation and validation logic beyond what type annotations alone can express.

Mark Day 2 complete in your [tracker](../index.html).